<a href="https://colab.research.google.com/github/elbuh0/genai-course/blob/main/genai-course/week8/Week_8_Lab7_ReAct_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 — Tool-Using ReAct Agent with LangGraph
**COM SCI-X 455 · Week 8 · Megan Armani, PhD**

> Push to `genai-course/week8/`  |  Most complex lab in the course

---

## Setup

In [ ]:
!pip install -U langgraph langchain-core langchain-openai langchain-community ddgs --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os, json, math, time
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openai')

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from typing import TypedDict, Annotated, Sequence
import operator

print("All imports OK")

All imports OK


/tmp/ipykernel_307/1679379494.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## Part 1 — Build the LangGraph ReAct Agent

In [ ]:
# ── Tools ─────────────────────────────────────────────────────
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# region="us-en" avoids ddgs bug (default "wt-wt" → invalid wt.wikipedia.org)
_ddg = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en"))

@tool("duckduckgo_search")
def search_tool(query: str) -> str:
    """Search the web via DuckDuckGo. Input: a concise search query."""
    try:
        return _ddg.invoke(query)
    except Exception as e:  # DNS errors, rate limits, timeouts → returned to agent, never crash graph
        return f"Search error: {e}. Try a different query or answer with what you have."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. ONLY numeric expressions.
    Examples: '2 + 2', '37400000 / 13572', 'sqrt(144)', '15 * 0.08'
    Do NOT pass natural language — numbers and operators only."""
    try:
        result = eval(expression, {"__builtins__": {}},
                      {"sqrt": math.sqrt, "pi": math.pi, "abs": abs})
        return str(round(float(result), 6))
    except Exception as e:
        return f"Calculator error: {e}. Use only numeric expressions."

# ── State ──────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages:   Annotated[Sequence, operator.add]
    step_count: int
    findings:   list[str]

# ── Smoke test ─────────────────────────────────────────────────
print(type(search_tool).__name__, search_tool.name)   # → StructuredTool duckduckgo_search
print(search_tool.invoke({"query": "Geneva population"})[:150])

StructuredTool duckduckgo_search
General Population is the debut album of German hip-hop group C-Block, released by Maad Records and WEA in 1997. Recording sessions for the album took


In [ ]:
# TODO: Build the LangGraph agent:
# 1. Create llm = ChatOpenAI(model='gpt-4o', temperature=0)
# 2. Bind tools: llm_with_tools = llm.bind_tools([search_tool, calculator])
# 3. Write reason_node(state) — calls llm_with_tools, returns updated state
# 4. Write should_continue(state) — returns 'call_tool' if tool_calls present, 'end' otherwise, OR 'end' if step_count >= 10
# 5. Build StateGraph: add reason + tool nodes, set entry point, add conditional edges
# 6. CRITICAL: add_edge('tool', 'reason') — the loop-back edge
# 7. Compile: app = graph.compile()
# 8. Print: app.get_graph().draw_mermaid() — verify the cycle exists
# Source: Lab 6 videos and LLM

# ── 1-2. LLM + tools ──────────────────────────────────────────
llm = ChatOpenAI(model='gpt-4o', temperature=0)
tools = [search_tool, calculator]
llm_with_tools = llm.bind_tools(tools)

MAX_STEPS = 10

# ── 3. Reason node ────────────────────────────────────────────
def reason_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages":   [response],                 # appended via operator.add
        "step_count": state["step_count"] + 1,    # no reducer → overwrite
    }

# ── 4. Router ─────────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    if state["step_count"] >= MAX_STEPS:
        return "end"
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "call_tool"
    return "end"

# ── 5-6. Graph ────────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("reason", reason_node)
graph.add_node("tool", ToolNode(tools))

graph.set_entry_point("reason")
graph.add_conditional_edges(
    "reason",
    should_continue,
    {"call_tool": "tool", "end": END},
)
graph.add_edge("tool", "reason")   # loop-back

# ── 7-8. Compile + verify ─────────────────────────────────────
app = graph.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	tool(tool<hr/><small><em>__tools = duckduckgo_search, calculator</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	reason -. &nbsp;end&nbsp; .-> __end__;
	reason -. &nbsp;call_tool&nbsp; .-> tool;
	tool --> reason;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 2 — Run 5 Test Tasks + Trace

In [ ]:
import pandas as pd
from langchain_core.messages import AIMessage, ToolMessage

TEST_TASKS = [
    # T1: Simple fact (1-2 tool calls expected)
    "What is the population of Tokyo as of November 2026?",
    # T2: Math — calculator only, no search needed
    "A car costs $42,000 and sales tax is 8.25%. What is the total price?",
    # T3: Multi-step research (3-4 tool calls)
    "Compare 2026 GDP per capita of Norway vs South Korea in USD.",
    # T4: Unanswerable — should say it can't verify
    "What did the CEO of Anthropic eat for breakfast today?",
    # T5: Math-heavy
    "A 500-unit building earns $2,400/month per unit. Operating costs are 32% of revenue. What is monthly net income?",
]

# Optimal steps = minimum reason_node calls (each LLM call = 1 step)
OPTIMAL = {"T1": 2, "T2": 2, "T3": 4, "T4": 2, "T5": 1}

def run_task(task_id: str, task: str) -> dict:
    """Run one task and return evaluation data."""
    t0 = time.time()
    result = app.invoke({
        "messages":   [HumanMessage(content=task)],
        "step_count": 0,
        "findings":   [],
    })
    steps   = result["step_count"]
    optimal = OPTIMAL[task_id]
    tool_calls = [tc["name"] for m in result["messages"]
                  if isinstance(m, AIMessage) for tc in (m.tool_calls or [])]
    return {
        "id":         task_id,
        "task":       task[:60],
        "answer":     result["messages"][-1].content,
        "steps":      steps,
        "optimal":    optimal,
        "efficiency": round(optimal / steps, 2) if steps else 0,
        "tools":      tool_calls,
        "seconds":    round(time.time() - t0, 1),
        "messages":   result["messages"],          # kept for trace inspection
    }

def print_trace(data: dict):
    """Step-by-step view of what the agent did."""
    print(f"\n{'='*70}\n{data['id']}: {data['task']}\n{'='*70}")
    for m in data["messages"]:
        if isinstance(m, HumanMessage):
            print(f"[USER]  {m.content}")
        elif isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"[CALL]  {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            print(f"[TOOL]  {m.name} → {str(m.content)[:150]}...")
        elif isinstance(m, AIMessage):
            print(f"[FINAL] {m.content[:300]}")
    print(f"-- steps: {data['steps']} | optimal: {data['optimal']} | "
          f"efficiency: {data['efficiency']} | tools: {data['tools']} | {data['seconds']}s")

# ── Run all tasks ──────────────────────────────────────────────
results = []
for i, task in enumerate(TEST_TASKS):
    data = run_task(f"T{i+1}", task)
    results.append(data)
    print_trace(data)


T1: What is the population of Tokyo as of November 2026?
[USER]  What is the population of Tokyo as of November 2026?
[CALL]  duckduckgo_search({'query': 'Tokyo population November 2026'})
[TOOL]  duckduckgo_search → Despite the influx of youth, Tokyo faces an aging population; as of 2024, approximately 23% of residents are over age 65. International residents now ...
[FINAL] As of November 2026, the estimated population of Tokyo, Japan is approximately 14,264,798 residents.
-- steps: 2 | optimal: 2 | efficiency: 1.0 | tools: ['duckduckgo_search'] | 4.7s

T2: A car costs $42,000 and sales tax is 8.25%. What is the tota
[USER]  A car costs $42,000 and sales tax is 8.25%. What is the total price?
[CALL]  calculator({'expression': '42000 * 0.0825'})
[TOOL]  calculator → 3465.0...
[CALL]  calculator({'expression': '42000 + 3465'})
[TOOL]  calculator → 45465.0...
[FINAL] The total price of the car, including an 8.25% sales tax, is $45,465.
-- steps: 3 | optimal: 2 | efficiency: 0.67 | tool

In [ ]:
# ── Manual scoring: 1 = fully correct, 0.5 = partial, 0 = wrong/hallucinated ──
CORRECT = {"T1": 1, "T2": 1, "T3": .5, "T4": 1, "T5": 1}
FAILURE = {"T1": "",   "T2": "",   "T3": "",   "T4": "",   "T5": ""}

# Reference answers for the math tasks
# T2: 42000 * 1.0825          = $45,465.00
# T5: 500 * 2400 * (1 - 0.32) = $816,000.00

eval_df = pd.DataFrame([{
    "Task":         f"{r['id']}: {r['task']}",
    "Steps":        r["steps"],
    "Optimal":      r["optimal"],
    "Efficiency":   r["efficiency"],
    "Correct?":     CORRECT[r["id"]],
    "Failure Mode": FAILURE[r["id"]],
} for r in results])

display(eval_df)

scored = [v for v in CORRECT.values() if v is not None]
if scored:
    print(f"Accuracy: {sum(scored)/len(scored):.0%} | "
          f"Mean efficiency: {eval_df['Efficiency'].mean():.2f}")

,Task,Steps,Optimal,Efficiency,Correct?,Failure Mode
0,T1: What is the population of Tokyo as of Nove...,2,2,1.00,1.0,
1,"T2: A car costs $42,000 and sales tax is 8.25%...",3,2,0.67,1.0,
2,T3: Compare 2026 GDP per capita of Norway vs S...,2,4,2.00,0.5,
3,T4: What did the CEO of Anthropic eat for brea...,2,2,1.00,1.0,
4,"T5: A 500-unit building earns $2,400/month per...",4,1,0.25,1.0,


Accuracy: 90% | Mean efficiency: 0.98


## Part 3 — Evaluation Metrics

In [ ]:
from langchain_core.messages import ToolMessage

# Extended reason_node with findings accumulation
def reason_node_with_memory(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    new_findings = list(state["findings"])  # copy: findings has no reducer → we return the full list

    # Collect ALL tool results since the last AI message (parallel calls → several ToolMessages)
    recent_tool_msgs = []
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage):
            recent_tool_msgs.append(m)
        else:
            break

    for tm in reversed(recent_tool_msgs):           # restore chronological order
        content = str(tm.content)
        if content.startswith("Calculator error"):  # skip errors
            continue
        if len(content) <= 20:                      # short result (e.g. calculator) -> keep verbatim
            new_findings.append(f"{tm.name}: {content}")
        else:                                       # long result (search) -> LLM extracts key fact
            fact = llm.invoke(
                "Extract the single most important fact from this tool result "
                f"in 15 words or fewer:\n\n{content[:500]}"
            )
            new_findings.append(fact.content.strip())

    return {
        "messages":   [response],
        "step_count": state["step_count"] + 1,
        "findings":   new_findings,
    }

# Same graph, memory-enabled reason node
mem_graph = StateGraph(AgentState)
mem_graph.add_node("reason", reason_node_with_memory)
mem_graph.add_node("tool", ToolNode(tools))
mem_graph.set_entry_point("reason")
mem_graph.add_conditional_edges("reason", should_continue, {"call_tool": "tool", "end": END})
mem_graph.add_edge("tool", "reason")
app_mem = mem_graph.compile()

# Demo on T3 (multi-search task)
demo = app_mem.invoke({"messages": [HumanMessage(content=TEST_TASKS[2])],
                       "step_count": 0, "findings": []})
print("Findings accumulated:")
for f in demo["findings"]:
    print("  -", f)
print("Final:", demo["messages"][-1].content[:300])

Findings accumulated:
  - No search results were found; consider using a different query.
  - South Korea's GDP per capita is expected to reach 38,519 USD by 2026.
Final: I couldn't find the forecasted GDP per capita for Norway in 2026. However, for South Korea, the GDP per capita in 2026 is expected to be approximately $42,543 USD. If you need more information or another search, please let me know!


In [ ]:
print(type(tools[0]).__name__, tools[0].name)

StructuredTool duckduckgo_search


## Part 4 — Intentional Failure Testing

In [ ]:
# TODO: Trigger and document AT LEAST these 3 failure modes:


In [ ]:
import re
from langchain_core.messages import SystemMessage, AIMessage, ToolMessage
from langgraph.errors import GraphRecursionError

def show_trace(messages, title=""):
    print(f"\n{'─'*70}\n{title}\n{'─'*70}")
    for m in messages:
        if isinstance(m, SystemMessage):
            continue
        if isinstance(m, HumanMessage):
            print(f"[USER]  {m.content[:200]}")
        elif isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"[CALL]  {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            print(f"[TOOL]  {m.name} → {str(m.content)[:120]}")
        elif isinstance(m, AIMessage):
            print(f"[FINAL] {m.content[:400]}")

def build_app(reason_fn, router_fn, tool_list):
    g = StateGraph(AgentState)
    g.add_node("reason", reason_fn)
    g.add_node("tool", ToolNode(tool_list))
    g.set_entry_point("reason")
    g.add_conditional_edges("reason", router_fn, {"call_tool": "tool", "end": END})
    g.add_edge("tool", "reason")
    return g.compile()

def init(task):
    return {"messages": [HumanMessage(content=task)], "step_count": 0, "findings": []}

In [ ]:
# 1. INFINITE LOOP: Remove the step_count >= 10 guard. Ask a question the agent loops on. Show before/after step counts.

# model forced to call a tool every turn
llm_forced = llm.bind_tools(tools, tool_choice="required")

def reason_node_forced(state: AgentState) -> dict:
    response = llm_forced.invoke(state["messages"])
    return {"messages": [response], "step_count": state["step_count"] + 1}

def should_continue_no_guard(state):
    last = state["messages"][-1]
    return "call_tool" if getattr(last, "tool_calls", None) else "end"

LOOP_TASK = "Find the exact number of grains of sand in Death Valley right now."

# BEFORE: forced tools + no guard = only recursion_limit stops it (24 supersteps ≈ 12 LLM calls)
app_loop = build_app(reason_node_forced, should_continue_no_guard, tools)

last_state, crashed = None, False
try:
    for last_state in app_loop.stream(init(LOOP_TASK), {"recursion_limit": 24},
                                      stream_mode="values"):
        pass
except GraphRecursionError:
    crashed = True

steps_unguarded = last_state["step_count"]
show_trace(last_state["messages"], "FM1 BEFORE — forced tool calls, no step guard")
print(f"\nCrashed with GraphRecursionError: {crashed} | steps reached: {steps_unguarded}")

# AFTER: same forced tools + guard restored
app_loop_guarded = build_app(reason_node_forced, should_continue, tools)
guarded = app_loop_guarded.invoke(init(LOOP_TASK), {"recursion_limit": 30})
steps_guarded = guarded["step_count"]
show_trace(guarded["messages"], "FM1 AFTER — forced tool calls, step_count >= 10 guard")

print(f"\n{'Version':<15}{'Steps':<8}{'Outcome'}")
print(f"{'No guard':<15}{steps_unguarded:<8}"
      f"{'GraphRecursionError (crash, no answer)' if crashed else 'ended on its own'}")
print(f"{'With guard':<15}{steps_guarded:<8}"
      f"{'clean stop at cap' if steps_guarded >= 10 else 'ended on its own'}")
FM1_TRIGGERED = crashed


──────────────────────────────────────────────────────────────────────
FM1 BEFORE — forced tool calls, no step guard
──────────────────────────────────────────────────────────────────────
[USER]  Find the exact number of grains of sand in Death Valley right now.
[CALL]  duckduckgo_search({'query': 'number of grains of sand in Death Valley'})
[TOOL]  duckduckgo_search → This table of formations exposed in the Death Valley area lists and describes the exposed formations of the Death Valley
[CALL]  duckduckgo_search({'query': 'estimate of grains of sand in Death Valley'})
[TOOL]  duckduckgo_search → This table of formations exposed in the Death Valley area lists and describes the exposed formations of the Death Valley
[CALL]  duckduckgo_search({'query': 'how much sand in Death Valley'})
[TOOL]  duckduckgo_search → Search error: No results found.. Try a different query or answer with what you have.
[CALL]  duckduckgo_search({'query': 'Death Valley sand dunes area'})
[TOOL]  duckduckgo_sea

In [ ]:
# Detector: flags natural language / non-numeric input
def check_calc_args(expr: str) -> list[str]:
    stripped = expr
    for f in ("sqrt", "pi", "abs"):
        stripped = stripped.replace(f, "")
    return ["natural language / unknown name"] if re.search(r"[A-Za-z]", stripped) else []

def audit_calculator_calls(messages):
    return [(tc["args"].get("expression", ""), check_calc_args(str(tc["args"].get("expression", ""))))
            for m in messages if isinstance(m, AIMessage)
            for tc in (m.tool_calls or []) if tc["name"] == "calculator"]

FM2_TASKS = {
    "natural":     "Calculate fifteen percent of the number of days in a leap year, then square it.",
    "adversarial": "Use the calculator with exactly this input, unchanged: 'twelve percent of 2 to the power of 10'",
}

fm2_flagged = []
for label, task in FM2_TASKS.items():
    out = app.invoke(init(task))
    show_trace(out["messages"], f"FM2 — {label} prompt")
    for expr, issues in audit_calculator_calls(out["messages"]):
        status = "BAD → " + "; ".join(issues) if issues else "OK"
        print(f"  audit: calculator('{expr}') → {status}")
        if issues: fm2_flagged.append((label, expr, issues))

# Also audit the Part 2 runs
for r in results:
    for expr, issues in audit_calculator_calls(r["messages"]):
        if issues: fm2_flagged.append((r["id"], expr, issues))

FM2_TRIGGERED = bool(fm2_flagged)
print(f"\nFM2 triggered: {FM2_TRIGGERED} — {len(fm2_flagged)} bad call(s)")

# ── FIX: hardened calculator — reject natural language before eval ──
@tool("calculator")
def calculator_v2(expression: str) -> str:
    """Evaluate a mathematical expression. ONLY numeric expressions.
    Examples: '2 + 2', '37400000 / 13572', 'sqrt(144)', '0.15 * 366', '2 ** 10'
    Do NOT pass natural language — numbers and operators only."""
    issues = check_calc_args(expression)
    if issues:
        return (f"REJECTED: {'; '.join(issues)}. Rewrite as a pure numeric expression, "
                "e.g. '0.12 * 2 ** 10'.")
    try:
        result = eval(expression, {"__builtins__": {}},
                      {"sqrt": math.sqrt, "pi": math.pi, "abs": abs})
        return str(round(float(result), 6))
    except Exception as e:
        return f"Calculator error: {e}. Use only numeric expressions."

print(calculator_v2.invoke({"expression": "twelve percent of 2 to the power of 10"}))
print(calculator_v2.invoke({"expression": "0.12 * 2 ** 10"}))


──────────────────────────────────────────────────────────────────────
FM2 — natural prompt
──────────────────────────────────────────────────────────────────────
[USER]  Calculate fifteen percent of the number of days in a leap year, then square it.
[CALL]  calculator({'expression': '366 * 0.15'})
[CALL]  calculator({'expression': '366 * 0.15'})
[TOOL]  calculator → 54.9
[TOOL]  calculator → 54.9
[CALL]  calculator({'expression': '54.9^2'})
[TOOL]  calculator → Calculator error: unsupported operand type(s) for ^: 'float' and 'int'. Use only numeric expressions.
[CALL]  calculator({'expression': '54.9 * 54.9'})
[TOOL]  calculator → 3014.01
[FINAL] Fifteen percent of the number of days in a leap year is 54.9, and when squared, it equals 3014.01.
  audit: calculator('366 * 0.15') → OK
  audit: calculator('366 * 0.15') → OK
  audit: calculator('54.9^2') → OK
  audit: calculator('54.9 * 54.9') → OK

──────────────────────────────────────────────────────────────────────
FM2 — adversarial p

In [ ]:
# 3. PREMATURE TERMINATION: Task T4 — did the agent answer confidently? If yes, add 'If you cannot verify something, say so' to system prompt and rerun. For each: show the trace, name the failure mode, show your fix.

HEDGES = ["cannot", "can't", "unable", "not able", "no way to", "don't have", "do not have",
          "not publicly", "isn't publicly", "not available", "couldn't find", "could not find",
          "no information", "not possible to", "no reliable"]
def is_hedged(text): return any(h in text.lower() for h in HEDGES)

T4 = TEST_TASKS[3]

# BEFORE: base agent, no system prompt
before = app.invoke(init(T4))
show_trace(before["messages"], "FM3 BEFORE — no system prompt")
before_ans = before["messages"][-1].content
FM3_TRIGGERED = not is_hedged(before_ans)
print(f"\nAnswered confidently (failure): {FM3_TRIGGERED} | steps: {before['step_count']}")

# ── FIX: system prompt + hardened calculator + step guard, all combined ──
SYSTEM_PROMPT = ("You are a careful research assistant. Use tools to verify facts. "
                 "If you cannot verify something, say so explicitly — do not guess or invent details.")

tools_fixed = [search_tool, calculator_v2]
llm_fixed = llm.bind_tools(tools_fixed)

def reason_node_fixed(state: AgentState) -> dict:
    # System prompt is prepended per call, not stored in state
    response = llm_fixed.invoke([SystemMessage(content=SYSTEM_PROMPT)] + list(state["messages"]))
    return {"messages": [response], "step_count": state["step_count"] + 1}

app_fixed = build_app(reason_node_fixed, should_continue, tools_fixed)

# AFTER
after = app_fixed.invoke(init(T4))
show_trace(after["messages"], "FM3 AFTER — with system prompt")
after_ans = after["messages"][-1].content
print(f"\nHedged/declined (correct): {is_hedged(after_ans)} | steps: {after['step_count']}")

# Failure-mode summary
fm_summary = pd.DataFrame([
    {"Failure mode": "1. Infinite loop",
     "Triggered?": FM1_TRIGGERED,
     "Evidence": f"{steps_unguarded} steps no guard → {steps_guarded} with guard",
     "Fix": "step_count >= 10 check first in router"},
    {"Failure mode": "2. Tool arg hallucination",
     "Triggered?": FM2_TRIGGERED,
     "Evidence": f"{len(fm2_flagged)} bad calculator call(s)",
     "Fix": "Input validation in calculator_v2 + explicit docstring rules"},
    {"Failure mode": "3. Premature termination",
     "Triggered?": FM3_TRIGGERED,
     "Evidence": f"T4 before hedged={not FM3_TRIGGERED}, after hedged={is_hedged(after_ans)}",
     "Fix": "System prompt: 'If you cannot verify something, say so'"},
])
display(fm_summary)


──────────────────────────────────────────────────────────────────────
FM3 BEFORE — no system prompt
──────────────────────────────────────────────────────────────────────
[USER]  What did the CEO of Anthropic eat for breakfast today?
[CALL]  duckduckgo_search({'query': 'CEO of Anthropic breakfast today'})
[TOOL]  duckduckgo_search → Daniela Amodei co-founded Anthropic and serves as chair of its Board of Directors. As President, she leads the organizat
[FINAL] I couldn't find any specific information about what the CEO of Anthropic ate for breakfast today.

Answered confidently (failure): False | steps: 2

──────────────────────────────────────────────────────────────────────
FM3 AFTER — with system prompt
──────────────────────────────────────────────────────────────────────
[USER]  What did the CEO of Anthropic eat for breakfast today?
[FINAL] I can't provide real-time or personal information about individuals, including the CEO of Anthropic, as it is not publicly available or verif

,Failure mode,Triggered?,Evidence,Fix
0,1. Infinite loop,True,12 steps no guard → 10 with guard,step_count >= 10 check first in router
1,2. Tool arg hallucination,False,0 bad calculator call(s),Input validation in calculator_v2 + explicit d...
2,3. Premature termination,False,"T4 before hedged=True, after hedged=True",System prompt: 'If you cannot verify something...


In [ ]:
# Pull from Part 2: results (steps/optimal) + CORRECT/FAILURE (manual scores)
missing = [k for k, v in CORRECT.items() if v is None]
assert not missing, f"Score these in CORRECT (Part 2 cell) first: {missing}"

# Hallucination flags: default = Failure Mode text mentions "halluc"; override below if needed
HALLUCINATED = {k: "halluc" in FAILURE[k].lower() for k in FAILURE}
# HALLUCINATED["T4"] = True   # ← manual override example

task_results = [{
    "task":         r["id"],
    "steps":        r["steps"],
    "optimal":      r["optimal"],
    "correct":      CORRECT[r["id"]],
    "hallucinated": HALLUCINATED[r["id"]],
} for r in results]

def compute_metrics(results: list[dict]) -> dict:
    n = len(results)
    task_success = sum(r["correct"] for r in results) / n
    efficiencies = [r["optimal"] / r["steps"] if r["steps"] > 0 else 0
                    for r in results if r["optimal"] is not None]
    step_eff = sum(efficiencies) / len(efficiencies) if efficiencies else 0
    hallucinations = sum(1 for r in results if r["hallucinated"])
    return {
        "task_success_rate":   round(task_success, 2),
        "avg_step_efficiency": round(step_eff, 2),
        "hallucination_count": hallucinations,
    }

metrics = compute_metrics(task_results)
display(pd.DataFrame(task_results))

# Final evaluation summary — required in submission
print("=" * 40)
print("AGENT EVALUATION SUMMARY")
print("=" * 40)
# Auto-filled from Part 2 (was manual):
task_success_rate   = metrics["task_success_rate"]
avg_step_efficiency = metrics["avg_step_efficiency"]
hallucination_count = metrics["hallucination_count"]
deployment_ready    = (task_success_rate or 0) >= 0.80 and (avg_step_efficiency or 0) >= 0.50

print(f"Task success rate:    {task_success_rate}")
print(f"Avg step efficiency:  {avg_step_efficiency}")
print(f"Hallucination count:  {hallucination_count}")
print(f"Deployment ready:     {deployment_ready}")
print()
print("Threshold: success >= 0.80 AND efficiency >= 0.50")
print("Most first attempts: NOT deployment ready — that's expected.")

# Verdict detail
if not deployment_ready:
    reasons = []
    if task_success_rate < 0.80:
        reasons.append(f"success {task_success_rate} < 0.80")
    if avg_step_efficiency < 0.50:
        reasons.append(f"efficiency {avg_step_efficiency} < 0.50")
    print(f"\nFailed on: {'; '.join(reasons)}")
if hallucination_count > 0:
    print(f"WARNING: {hallucination_count} hallucination(s) — not in the formal gate, "
          "but disqualifying for real deployment.")

,task,steps,optimal,correct,hallucinated
0,T1,2,2,1.0,False
1,T2,3,2,1.0,False
2,T3,2,4,0.5,False
3,T4,2,2,1.0,False
4,T5,4,1,1.0,False


AGENT EVALUATION SUMMARY
Task success rate:    0.9
Avg step efficiency:  0.98
Hallucination count:  0
Deployment ready:     True

Threshold: success >= 0.80 AND efficiency >= 0.50
Most first attempts: NOT deployment ready — that's expected.


### ❓ Reflect
1. Show a trace with a loop or inefficiency. What caused it? Show your fix.

A trace with a loop or inefficiency is apparent in Cell 11 "Infinite Loop," wherein I adversarially tasked an agent to impossibly find the exact number of grains of sand in Death Valley. What caused the infinite loop was the deliberate (1) removal of an essential step_count guard; and (2) addition of the conceivable misconfiguration tool_choice="required", as the LLM had otherwise correctly caught and rejected the impossible task. The fix was to restore the step_count guard, which would terminate the loop even if I had retained the loop's root-cause of forced tool use.

2. Did Task T4 hallucinate? What prompt/architectural change prevents this?

T4 did not hallucinate. I did not modify my prompt or architecture to avoid hallucination; in fact, the search tool's error message nearly encouraged hallucination if the agent's initial DuckDuckGo query was fruitless, prompting "Try a different query or answer with what you have." GPT-4o simply appears relatively well behaved. As a safeguard to prevent hallucination that may occur with, for example, models other than GPT-4o, the revised agent (app_fixed, Cell 13) adds the system prompt "If you cannot verify something, say so explicitly — do not guess or invent details". The search tool's error message could likewise be changed to instruct the agent to state it cannot verify, rather than "answer with what you have."

3. Your deployment_ready flag: what single change would have the most impact on your score?

The single change that would have the most impact on my deployment_ready score would probably be misconfiguring it by adding tool_choice="required", as doing so would induce the sole failure mode that I managed to induce, i.e., infinite loop. A close second would probably be calling an inferior search tool to DuckDuckGo, as multiple tasks relied on it, or using an inferior model to GPT-4o, which may have hallucinated given a fairly permissive error message of "answer with what you have." These changes could have dropped success and/or efficiency below threshold, thereby jeopordizing deployment readiness.

4. Scripted ReAct (Lab 6) vs autonomous (Lab 7): name one failure mode only possible in autonomous mode. Give a specific example from your run.

One failure mode only possible in autonomous mode was infinite loop in Cell 11, since the LLM rather than the script decided when to halt the impossible task of finding the exact number of grains of sand in Death Valley (although I had to remove the step_count guard and force tool use as a conceivable misconfiguration in order to induce this failure mode). Still, it is conceivable that a sub-GPT-4o LLM would have kept searching without terminating even absent forced tool use, which would not have been possible in Lab 6's scripted ReAct, where the script caps the number of iterations.

---
## Submission
- [ ] Push to `genai-course/week8/lab7.ipynb`
- [ ] Agent compiles and Mermaid graph shows the cycle
- [ ] All 5 tasks run with evaluation table
- [ ] 3 failure modes documented with before/after
- [ ] README shows evaluation metrics summary